<a href="https://colab.research.google.com/github/Kanchanajaddu/GEN_AI-exercises/blob/main/Q%26Achatbot(w2_d5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
#given question:
#Build Q&A chatbot app:
#Ingest a dataset (PDF/book/docs).
#Embed & store chunks.
#Query → retrieve → LLM → answer.

In [ ]:
#open ai api key
import os
openai_api_key = input("Please enter your OpenAI API key (e.g., sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx): ")
os.environ["OPENAI_API_KEY"] = openai_api_key
print("OpenAI API key configured successfully.")

In [7]:
pip install langchain openai tiktoken pypdf faiss-cpu

In [8]:
pip install langchain-community langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-co

In [9]:
#upload documents
from google.colab import files
import os
def upload_files():
    uploaded = files.upload()
    file_paths = []
    for filename in uploaded.keys():
        print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')
        file_paths.append(filename)
    return file_paths
corpus_files = upload_files()


Saving health care policy document 3.pdf to health care policy document 3.pdf
User uploaded file "health care policy document 3.pdf" with length 438003 bytes


In [10]:
#loading and processing the documents
from langchain_community.document_loaders import PyPDFLoader, TextLoader, UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
def load_documents(file_paths):
    docs = []
    for file_path in file_paths:
        _, file_extension = os.path.splitext(file_path)
        if file_extension == '.pdf':
            loader = PyPDFLoader(file_path)
        elif file_extension == '.txt':
            loader = TextLoader(file_path)
        elif file_extension == '.md':
            loader = TextLoader(file_path)
        else:
            print(f"Skipping unsupported file type: {file_path}")
            continue
        docs.extend(loader.load())
    return docs

def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap  = 200,
        length_function = len,
    )
    return text_splitter.split_documents(documents)

# Load and split documents
raw_documents = load_documents(corpus_files)
text_chunks = split_documents(raw_documents)

print(f"Loaded {len(raw_documents)} raw documents and split into {len(text_chunks)} text chunks.")

Loaded 27 raw documents and split into 125 text chunks.


In [11]:
#create embeddings and vector store
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(text_chunks, embeddings)

print("Vector store created successfully using FAISS.")

Vector store created successfully using FAISS.


In [14]:
#set up the knowledge assistant
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
llm = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# Create the RAG chain using LCEL
qa_chain = (
    { "context": itemgetter("question") | vectorstore.as_retriever(), "question": itemgetter("question") }
    | prompt
    | llm
)

print("Knowledge Assistant (RAG chain) initialized. You can now ask questions!")

Knowledge Assistant (RAG chain) initialized. You can now ask questions!


In [16]:
#asking questions
from IPython.display import display, Markdown

def ask_question(question):
    if question:
        print(f"\nQuestion: {question}")
        result = qa_chain.invoke({"question": question})
        answer = result.content
        print(f"\nAnswer: {answer}")
    else:
        print("Please enter a question.")
while True:
    user_question = input("Your question (type 'exit' to quit): ")
    if user_question.lower() == 'exit':
        print("Exiting Q&A session.")
        break
    ask_question(user_question)

Your question (type 'exit' to quit): what is the main topic of the document

Question: what is the main topic of the document

Answer: Health Insurance Policy - Retail
Your question (type 'exit' to quit): exit
Exiting Q&A session.
